# Lesson 3.1 — From Target Pose to Goal Configuration

The committed target is a **pose**; the planner needs a **configuration**. We convert via Module 5 IK (`to_configuration`) and verify with Module 4 FK, and confirm the two failure-relevant facts: unreachable → None, two elbow solutions.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception, understand,
    to_configuration, ik_2link, fk_xy, P2, T2, REACH_MAX)
checks = []
world = GreenhouseWorld.demo_row(n=6, seed=1)
dets = model_perception(world, rng=np.random.default_rng(7))
_, target = understand(dets, world)
print('committed target:', target['id'], np.round(target['xy'],3))


committed target: F3 [0.447 0.152]


### IK -> configuration, verified by FK round-trip

In [2]:
q_goal = to_configuration(target)
back = fk_xy(P2, T2, q_goal)
err = float(np.linalg.norm(back - target['xy']))
print('q_goal:', np.round(q_goal,4), '| FK(q_goal):', np.round(back,4), '| round-trip err: %.2e' % err)
checks.append(q_goal is not None)
checks.append(err < 1e-9)                 # FK verifies the configuration reaches the target


q_goal: [-0.3558  1.6844] | FK(q_goal): [0.4469 0.1519] | round-trip err: 6.21e-17


### Unreachable target -> None (no fudging)

In [3]:
far = {'xy': np.array([REACH_MAX + 0.3, 0.0])}
q_far = to_configuration(far)
print('unreachable pose -> to_configuration:', q_far)
checks.append(q_far is None)


unreachable pose -> to_configuration: None


### Two solutions: elbow up vs elbow down (both reach the target)

In [4]:
x, y = float(target['xy'][0]), float(target['xy'][1])
q_up = ik_2link(x, y, elbow='up'); q_dn = ik_2link(x, y, elbow='down')
up_ok = np.linalg.norm(fk_xy(P2,T2,q_up) - target['xy']) < 1e-9
dn_ok = np.linalg.norm(fk_xy(P2,T2,q_dn) - target['xy']) < 1e-9
print('elbow up  q:', np.round(q_up,3), 'reaches target:', up_ok)
print('elbow down q:', np.round(q_dn,3), 'reaches target:', dn_ok)
checks.append(up_ok and dn_ok and not np.allclose(q_up, q_dn))  # two distinct valid solutions
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


elbow up  q: [-0.356  1.684] reaches target: True
elbow down q: [ 1.011 -1.684] reaches target: True
4/4 checks passed.
All checks passed.
